# ДЗ №4: Выделение ключевых особенностей / Homework 4: Feature Detection
**ID: FD-04 · 100 баллов / points.**

Срок берётся из таблицы сроков для вашей группы. Штраф рассчитывается при проверке по времени принятой версии в Google Drive, не по времени запуска ноутбука. / The deadline comes from the group deadline sheet. The checker applies the late penalty using the accepted Google Drive submission timestamp, not the notebook execution time.

[Таблица сроков / Deadlines](https://docs.google.com/spreadsheets/d/15N5GGHujXPAzRNuyOfmooT7A7sKhXgrMfWHkUkvYYiM/edit)
[Практика / Practice](Practice_RU_EN.ipynb)

Заполните личные данные, реализуйте четыре функции и выполните Restart & Run All. Сдайте один .ipynb с кодом, выводами и объяснением. Инструкции даны на двух языках; ответ можно написать на любом из них. Не меняйте тесты и ID. / Fill in your identity, implement four functions, restart and run all cells. Submit one .ipynb with code, outputs and explanation. Instructions are bilingual; write your answer in either language. Preserve tests and ID.

## Данные / Data
BSDS500: одна запись — реальная фотография; входные признаки — яркость и её производные; целевая разметка — границы, отмеченные людьми. Это не разметка углов или прямых. / Each observation is a real photograph. Inputs are intensity and its derivatives; targets are human boundary annotations, not corner or line labels.

[Официальный источник / Official source](https://www2.eecs.berkeley.edu/Research/Projects/CS/vision/grouping/resources.html). Архив около 67 MiB, проверяется SHA256. / About 67 MiB, verified by SHA256.
Train: 100075, 108073, 118035. Val: 101085, 101087, 102061. Test: 100007, 100039, 100099.
Параметры выбираются только на val и фиксируются до test. Нельзя подбирать пороги по test или смешивать исходные разбиения. / Select parameters only on validation and freeze them before testing. Do not tune on test or mix the original splits.
[Условия BSDS / Terms](https://www2.eecs.berkeley.edu/Research/Projects/CS/vision/bsds/): некоммерческие исследовательские и образовательные цели, исходные права сохраняются. / Non-commercial research and education; original rights apply.
Cite Arbelaez, Maire, Fowlkes, Malik, TPAMI 2011; Martin et al., ICCV 2001.

CPU достаточно. Python 3.10+, NumPy, OpenCV, SciPy. / CPU is sufficient. Python 3.10+, NumPy, OpenCV, SciPy.


In [ ]:
# При необходимости / If needed:
# %pip install "numpy>=1.26,<3" "opencv-python-headless>=4.10,<6" "scipy>=1.11,<2"
student_first = ""
student_last = ""
student_group = ""
assignment_id = "FD-04"


In [ ]:
from pathlib import Path
import hashlib, json, tarfile, urllib.request
import cv2
import numpy as np
from scipy.io import loadmat
from scipy.spatial import cKDTree
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import maximum_bipartite_matching

DATA_URL = 'https://www2.eecs.berkeley.edu/Research/Projects/CS/vision/grouping/BSR/BSR_bsds500.tgz'
ARCHIVE_SHA256 = '97e49d31764f3912f0c4122707d53062ac9e783ba0f095e447a4d53c1a41af8e'
SPLITS = {'train':['100075','108073','118035'], 'val':['101085','101087','102061'], 'test':['100007','100039','100099']}

def prepare_data(cache):
    cache = Path(cache); cache.mkdir(parents=True, exist_ok=True)
    archive = cache / 'BSR_bsds500.tgz'
    if not archive.exists():
        temp = archive.with_suffix('.download')
        try:
            with urllib.request.urlopen(DATA_URL, timeout=120) as src, temp.open('wb') as dst:
                import shutil
                shutil.copyfileobj(src, dst)
        except Exception as exc:
            raise RuntimeError('BSDS500 unavailable. Retry official download; do not replace with synthetic data.') from exc
        if hashlib.sha256(temp.read_bytes()).hexdigest() != ARCHIVE_SHA256:
            raise ValueError('Unexpected archive checksum; keep the file for inspection and do not extract.')
        temp.replace(archive)
    if hashlib.sha256(archive.read_bytes()).hexdigest() != ARCHIVE_SHA256:
        raise ValueError('Archive checksum mismatch')
    root = cache / 'BSR/BSDS500/data'
    with tarfile.open(archive) as tar:
        for split, ids in SPLITS.items():
            for iid in ids:
                for kind, ext in [('images','jpg'), ('groundTruth','mat')]:
                    name = f'BSR/BSDS500/data/{kind}/{split}/{iid}.{ext}'
                    dest = cache / name
                    member = tar.getmember(name)
                    if not member.isfile(): raise ValueError('Expected regular file')
                    source = tar.extractfile(member).read()
                    if dest.exists():
                        if hashlib.sha256(dest.read_bytes()).digest() != hashlib.sha256(source).digest():
                            raise ValueError(f'Cached dataset file changed: {dest}')
                    else:
                        dest.parent.mkdir(parents=True, exist_ok=True)
                        dest.write_bytes(source)
    return root

def load_example(root, split, iid):
    if iid not in SPLITS[split]: raise ValueError('Use the frozen teaching subset')
    bgr = cv2.imread(str(Path(root)/'images'/split/f'{iid}.jpg'))
    if bgr is None: raise ValueError('Unreadable image')
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    records = loadmat(Path(root)/'groundTruth'/split/f'{iid}.mat')['groundTruth'].ravel()
    annotations = [np.asarray(r['Boundaries'][0,0], dtype=bool) for r in records]
    assert gray.dtype == np.uint8 and gray.ndim == 2
    assert annotations and all(a.shape == gray.shape and a.any() for a in annotations)
    return rgb, gray, annotations

def gradients(gray, operator='sobel'):
    if gray.ndim != 2 or gray.dtype != np.uint8: raise ValueError('Expected grayscale uint8')
    src = gray.astype(np.float32) / 255
    if operator == 'sobel':
        gx = cv2.Sobel(src, cv2.CV_32F, 1, 0, ksize=3, scale=1/8, borderType=cv2.BORDER_REFLECT_101)
        gy = cv2.Sobel(src, cv2.CV_32F, 0, 1, ksize=3, scale=1/8, borderType=cv2.BORDER_REFLECT_101)
    elif operator == 'scharr':
        gx = cv2.Scharr(src, cv2.CV_32F, 1, 0, scale=1/32, borderType=cv2.BORDER_REFLECT_101)
        gy = cv2.Scharr(src, cv2.CV_32F, 0, 1, scale=1/32, borderType=cv2.BORDER_REFLECT_101)
    else: raise ValueError('Unknown derivative operator')
    return gx, gy, np.hypot(gx,gy), np.arctan2(gy,gx)

def canny(gray, low=50, high=100, sigma=1.0):
    if gray.ndim != 2 or gray.dtype != np.uint8: raise ValueError('Expected grayscale uint8')
    if not 0 <= low < high or sigma <= 0: raise ValueError('Invalid Canny parameters')
    smooth = cv2.GaussianBlur(gray, (0,0), sigmaX=float(sigma), borderType=cv2.BORDER_REFLECT_101)
    return cv2.Canny(smooth, float(low), float(high), apertureSize=3, L2gradient=True).astype(bool)

def boundary_prf(pred, truth, tolerance=2.0):
    """Teaching metric: maximum-cardinality one-to-one pixel matching.
    Euclidean tolerance in original-image pixels. Not the official BSDS metric.
    Empty/empty gives (1,1,1); one empty gives (0,0,0).
    """
    pred, truth = np.asarray(pred,dtype=bool), np.asarray(truth,dtype=bool)
    if pred.shape != truth.shape or pred.ndim != 2 or tolerance < 0: raise ValueError('Invalid boundary maps')
    a, b = np.argwhere(pred), np.argwhere(truth)
    if not len(a) and not len(b): return np.ones(3)
    if not len(a) or not len(b): return np.zeros(3)
    neighbors = cKDTree(b).query_ball_point(a, tolerance)
    sizes = np.array([len(x) for x in neighbors])
    rr = np.repeat(np.arange(len(a)), sizes)
    cc = np.array([j for row in neighbors for j in row],dtype=int)
    graph = csr_matrix((np.ones(len(cc),dtype=np.int8),(rr,cc)),shape=(len(a),len(b)))
    count = np.count_nonzero(maximum_bipartite_matching(graph,perm_type='column') >= 0)
    precision, recall = count/len(a), count/len(b)
    f1 = 2*precision*recall/(precision+recall) if count else 0
    return np.array([precision,recall,f1])

def evaluate(root, split, params):
    rows=[]
    for iid in SPLITS[split]:
        _, gray, anns = load_example(root,split,iid)
        edges = canny(gray,**params)
        prf = np.mean([boundary_prf(edges,a) for a in anns],axis=0)
        rows.append({'image':iid,'annotators':len(anns),'precision':float(prf[0]),'recall':float(prf[1]),'f1':float(prf[2])})
    return rows

def corners(gray, harris=False):
    # Unified point selection: same qualityLevel and minDistance for both scores.
    points = cv2.goodFeaturesToTrack(gray, maxCorners=80, qualityLevel=.02, minDistance=8,
                                    blockSize=3, useHarrisDetector=harris, k=.04)
    return np.empty((0,2),dtype=np.float32) if points is None else points.reshape(-1,2)

def line_segments(gray):
    edges = canny(gray)
    lines = cv2.HoughLinesP(edges.astype(np.uint8)*255,1,np.pi/180,threshold=35,minLineLength=40,maxLineGap=8)
    return np.empty((0,4),dtype=int) if lines is None else lines.reshape(-1,4)

DATA_ROOT = prepare_data(Path("bsds500_cache"))


## 1. Модуль и направление / Magnitude and direction — 20
`gradient_polar(gx, gy)` принимает знаковые производные float32 одинаковой формы (H,W). Верните пару массивов: sqrt(gx²+gy²), atan2(gy,gx), угол в радианах. Для (0,0) угол 0. Входы не изменять. / Given signed float32 derivatives of equal shape, return magnitude and orientation in radians. The angle at (0,0) is zero. Preserve inputs.
Знаки производных пригодятся ещё до того, как мы начнём спорить о красивой картинке. / Derivative signs matter before we start arguing about a pretty picture.

In [ ]:
def gradient_polar(gx, gy):
    raise NotImplementedError("Ваш код / Your code")


## 2. Canny с явным сглаживанием / Canny with explicit smoothing — 30
`detect_edges(gray, low, high, sigma)` принимает двумерное uint8-изображение. Проверьте `0 <= low < high`, `sigma > 0`, конечность параметров; иначе ValueError. / Accept a 2D uint8 image. Reject invalid or nonfinite parameters with ValueError.
Примените GaussianBlur с kernel (0,0), sigmaX=sigma, BORDER_REFLECT_101, затем Canny с apertureSize=3, L2gradient=True. Верните bool-маску той же формы. Не изменяйте gray. / Apply these exact smoothing and Canny settings, return a bool mask of the same shape, preserve gray.

In [ ]:
def detect_edges(gray, low, high, sigma):
    raise NotImplementedError("Ваш код / Your code")


## 3. Два отклика угла / Two corner scores — 25
`corner_scores(a, b, c, k)` получает элементы матрицы вторых моментов M=[[a,b],[b,c]] как float64-массивы одинаковой формы, k=0.04. Верните `(harris, shi)`:
Harris = a*c-b²-k*(a+c)²; Shi–Tomasi = (a+c-sqrt((a-c)²+4b²))/2.
Не обрезайте отрицательный Harris и не нормируйте карты. Входы не изменять. / Return the Harris response and smallest eigenvalue. Preserve negative Harris scores, do not normalize or mutate inputs.
Здесь вычисляем оценки, а не список точек. / These are response maps, not a selected point list.

In [ ]:
def corner_scores(a, b, c, k=0.04):
    raise NotImplementedError("Ваш код / Your code")


## 4. Выбор параметров / Parameter selection — 25
`select_config(validation_f1)` принимает конечный float64-массив (C,N) со значениями [0,1]: строки — конфигурации, столбцы — validation-изображения. C,N >= 1. Верните Python int — индекс максимального среднего F1 по изображениям. При равенстве выбирайте первую строку. / Return the Python int index with the largest mean validation F1; ties select the first row.
Не усредняйте по конфигурациям. Test здесь появляться не должен. / Average over images, not configurations. Test results do not belong here.

In [ ]:
def select_config(validation_f1):
    raise NotImplementedError("Ваш код / Your code")


## Проверка / Checks
Тесты используют фотографии и разметку BSDS500. Числа F1 — учебное сопоставление один к одному в радиусе 2 px, не официальный BSDS score. / Tests use BSDS500 photos and annotations. F1 uses teaching one-to-one matching within 2 pixels, not the official BSDS score.
Каждая задача оценивается независимо; повторный запуск сбрасывает балл. Недоступность данных — ошибка инфраструктуры, а не ноль за работу. / Tasks are scored independently; reruns reset the score. Unavailable data is an infrastructure failure, not a zero grade.


In [ ]:
# Подготовка эталонов до подсчёта баллов / Prepare references before scoring
photos = [load_example(DATA_ROOT, 'train', iid)[1] for iid in SPLITS['train']]
CONFIGS = [{'low':l,'high':h,'sigma':s} for l,h in [(20,40),(50,100),(100,200)] for s in [1.,1.6]]
validation_f1 = np.array([[r['f1'] for r in evaluate(DATA_ROOT,'val',p)] for p in CONFIGS])
def check_array(got, expected):
    assert isinstance(got,np.ndarray) and got.shape==expected.shape
    assert np.isfinite(got).all()
    np.testing.assert_allclose(got,expected,rtol=1e-5,atol=1e-7)
def call_preserving(fn,*args):
    copies=[x.copy() if isinstance(x,np.ndarray) else x for x in args]
    result=fn(*args)
    for x,y in zip(args,copies):
        if isinstance(x,np.ndarray): np.testing.assert_array_equal(x,y)
    return result

def test1():
    for gray in photos:
        gx,gy,mag,angle=gradients(gray)
        got=call_preserving(gradient_polar,gx,gy)
        assert isinstance(got,(tuple,list)) and len(got)==2
        check_array(got[0],mag);check_array(got[1],angle)
def test2():
    for gray in photos:
        for p in CONFIGS:
            got=call_preserving(detect_edges,gray,p['low'],p['high'],p['sigma'])
            assert got.dtype==np.bool_
            check_array(got,canny(gray,**p))
    for l,h,s in [(-1,50,1),(50,50,1),(100,50,1),(20,40,0),(20,40,float('nan')),(20,float('inf'),1)]:
        try: detect_edges(photos[0],l,h,s)
        except ValueError: pass
        else: raise AssertionError('Expected ValueError')
def test3():
    for gray in photos:
        gx,gy,_,_=gradients(gray);gx=gx.astype(float);gy=gy.astype(float)
        a,b,c=[cv2.GaussianBlur(x,(0,0),1.2) for x in (gx*gx,gx*gy,gy*gy)]
        got=call_preserving(corner_scores,a,b,c,.04)
        check_array(got[0],a*c-b*b-.04*(a+c)**2)
        matrix=np.stack([np.stack([a,b],axis=-1),np.stack([b,c],axis=-1)],axis=-2)
        check_array(got[1],np.linalg.eigvalsh(matrix)[...,0])
def test4():
    for values in [validation_f1,validation_f1[::-1].copy(),validation_f1[:,[2,0,1]],np.repeat(validation_f1[:1],2,axis=0)]:
        got=call_preserving(select_config,values)
        expected=max(range(len(values)),key=lambda i:sum(values[i])/len(values[i]))
        assert type(got) is int and got==expected

raw_score=0
for number,points,test in [(1,20,test1),(2,30,test2),(3,25,test3),(4,25,test4)]:
    try: test();earned=points;detail='OK'
    except Exception as exc: earned=0;detail=f'{type(exc).__name__}: {exc}'
    raw_score+=earned
    print(f'T{number}: {earned}/{points} — {detail}')
print(f'Исходный балл / Raw score: {raw_score}/100')


## Короткий эксперимент / Short experiment
После реализации функций выберите конфигурацию на validation, зафиксируйте параметры, затем один раз оцените test предоставленной функцией evaluate. Сохраните вывод ячейки ниже. / Select a configuration on validation, freeze its parameters, then evaluate test once with the supplied evaluate function. Preserve the output below.
Добавьте 5–8 предложений: почему текстура даёт лишние края; чем оценки угла отличаются от границ; почему test не участвует в настройке. Укажите выбранные параметры и средний test F1. Текст проверяется преподавателем, отдельных автоматических баллов за него нет. / Add 5–8 sentences explaining texture edges, corner scores versus boundaries, and why test cannot guide tuning. Report chosen parameters and mean test F1. The teacher reviews the text; it carries no separate automated points.

In [ ]:
# Выполните после реализации / Run after implementation
if raw_score == 100:
    selected = select_config(validation_f1)
    frozen_params = dict(CONFIGS[selected])
    print('Frozen parameters:', frozen_params)
    test_rows = evaluate(DATA_ROOT,'test',frozen_params)
    print(json.dumps({'parameters':frozen_params,'test':test_rows,'mean_test_f1':float(np.mean([r['f1'] for r in test_rows]))},indent=2))
else:
    print('Сначала исправьте функции / Fix the functions first')


Ваш разбор / Your analysis:

## Итог для проверки / Checker result
JSON ниже содержит исходный балл без штрафа. При проверке преподаватель повторно выполняет доверенные тесты; сохранённый студентом балл не является доказательством. / The JSON reports the raw score before penalties. The teacher reruns trusted tests; a student-saved score is not evidence.

In [ ]:
identity_ok=all(isinstance(v,str) and v.strip() for v in (student_first,student_last,student_group))
if not identity_ok: print('Заполните имя, фамилию и группу / Fill in your name and group')
print(json.dumps({'name':' '.join(str(v).strip() for v in (student_first,student_last)),'group':str(student_group).strip(),'assignment':'FD-04','score':float(raw_score if identity_ok else 0)},ensure_ascii=False,allow_nan=False))
